# Model-ready M0 EDA — no weather, no split, no model fitting

This notebook audits the M0 outputs built by `03_feature_engineering.ipynb`. P95 is the primary outcome; P90 is a sensitivity outcome on the same ordered eligible household-date population. This notebook describes target prevalence, time and household coverage, feature missingness/distributions, correlations, and relative-versus-absolute demand associations. It also audits the frozen feature-specific quality policy applied from the 03B masking experiment. It does not select a split or train a model.


## Stage 0 — reproducible inputs and output locations

Input datasets and JSON dtype contracts are checked against the feature-engineering run manifest. The frozen target configuration and 03B feature-quality policy provenance are reconciled before analysis. Report tables are CSV; charts are saved as PNG and also displayed in their output cells.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
if not (PROJECT_ROOT / "data").is_dir() or not (PROJECT_ROOT / "notebooks").is_dir():
    raise RuntimeError(f"Could not identify project root: {PROJECT_ROOT}")
PROCESSED = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "03_model_ready_eda"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures" / "03_model_ready_eda"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PATH = PROCESSED / "04_feature_dataset_m0.csv"
CANONICAL_PATH = PROCESSED / "05_model_ready_m0.csv"
P95_PATH = PROCESSED / "05_model_ready_m0_p95.csv"
P90_PATH = PROCESSED / "05_model_ready_m0_p90.csv"
PATHS = {"feature_dataset": FEATURE_PATH, "canonical": CANONICAL_PATH, "p95": P95_PATH, "p90": P90_PATH}
CONTRACT_PATHS = {
    "feature_dataset": PROCESSED / "04_feature_dataset_m0_dtype_contract.json",
    "canonical": PROCESSED / "05_model_ready_m0_dtype_contract.json",
    "p95": PROCESSED / "05_model_ready_m0_p95_dtype_contract.json",
    "p90": PROCESSED / "05_model_ready_m0_p90_dtype_contract.json",
}
FEATURE_MANIFEST_PATH = PROJECT_ROOT / "reports" / "tables" / "04_feature_engineering" / "04_run_manifest.json"
STAGE3_MANIFEST_PATH = PROJECT_ROOT / "reports" / "tables" / "03_household_day_target" / "03_run_manifest.json"
TARGET_CONFIG_PATH = PROJECT_ROOT / "reports" / "tables" / "03a_target_design_experiments" / "03a_selected_target_config.json"
FEATURE_QUALITY_CANDIDATE_PATH = PROJECT_ROOT / "reports" / "tables" / "03b_historical_feature_quality_experiments" / "03b_feature_quality_policy_candidates.csv"
FEATURE_QUALITY_CONFIG_PATH = PROJECT_ROOT / "reports" / "tables" / "03b_historical_feature_quality_experiments" / "03b_selected_feature_quality_config.json"
FEATURE_QUALITY_MANIFEST_PATH = PROJECT_ROOT / "reports" / "tables" / "03b_historical_feature_quality_experiments" / "03b_run_manifest.json"
SPLIT_MASTER_PATH = PROCESSED / "04_master_split.csv"
SPLIT_MANIFEST_PATH = PROJECT_ROOT / "reports" / "tables" / "04_split_design" / "04_split_design_manifest.json"
CSV_CHUNK_SIZE = 100_000

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size): digest.update(block)
    return digest.hexdigest()

def read_contract_csv(path, contract, usecols=None, chunksize=None):
    selected = list(contract["columns"]) if usecols is None else list(usecols)
    dates = [column for column in contract.get("date_columns", []) if column in selected]
    dtypes = {column: dtype for column, dtype in contract["columns"].items() if column in selected and dtype != "datetime"}
    return pd.read_csv(path, usecols=selected, dtype=dtypes or None, parse_dates=dates or None,
                       true_values=contract["reader"]["true_values"], false_values=contract["reader"]["false_values"],
                       keep_default_na=contract["reader"]["keep_default_na"], na_values=contract["reader"]["na_values"],
                       encoding=contract["encoding"], sep=contract["separator"], chunksize=chunksize, low_memory=False)

def save_table(frame, filename, title=None):
    path = TABLE_DIR / filename
    if path.suffix.lower() != ".csv": raise ValueError(f"EDA report tables must be CSV: {path}")
    frame.to_csv(path, index=False, encoding="utf-8", na_rep="")
    if title: print(f"{title}: {path.name}")
    display(frame.head(30))
    return path

def save_display_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print(f"Saved figure: {path}")

required = [*PATHS.values(), *CONTRACT_PATHS.values(), FEATURE_MANIFEST_PATH, STAGE3_MANIFEST_PATH, TARGET_CONFIG_PATH, FEATURE_QUALITY_CANDIDATE_PATH, FEATURE_QUALITY_CONFIG_PATH, FEATURE_QUALITY_MANIFEST_PATH, SPLIT_MASTER_PATH, SPLIT_MANIFEST_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError(f"Model-ready EDA inputs missing: {missing}")

feature_manifest = read_json(FEATURE_MANIFEST_PATH)
stage3_manifest = read_json(STAGE3_MANIFEST_PATH)
target_config = read_json(TARGET_CONFIG_PATH)
quality_policy = feature_manifest["feature_quality_policy"]
if Path(quality_policy["source_candidate_table"]).resolve() != FEATURE_QUALITY_CANDIDATE_PATH.resolve() or Path(quality_policy["source_manifest"]).resolve() != FEATURE_QUALITY_MANIFEST_PATH.resolve() or Path(quality_policy["configuration_path"]).resolve() != FEATURE_QUALITY_CONFIG_PATH.resolve():
    raise RuntimeError("Feature-quality source paths differ from the expected project artifacts.")
feature_quality_candidates = pd.read_csv(FEATURE_QUALITY_CANDIDATE_PATH, low_memory=False)
feature_quality_manifest = read_json(FEATURE_QUALITY_MANIFEST_PATH)
feature_quality_config = read_json(FEATURE_QUALITY_CONFIG_PATH)
split_manifest = read_json(SPLIT_MANIFEST_PATH)
if feature_manifest.get("target_configuration_id") != target_config.get("configuration_id"):
    raise RuntimeError("Feature-engineering target config ID differs from frozen 03A.")
if stage3_manifest.get("target_configuration_id") != target_config.get("configuration_id"):
    raise RuntimeError("Stage 3 target config ID differs from frozen 03A.")
if feature_manifest.get("target_policy_status") != "frozen_primary_research_policy":
    raise RuntimeError("Model-ready EDA requires a frozen target policy.")
if quality_policy.get("configuration_id") != feature_quality_config.get("configuration_id"):
    raise RuntimeError("Feature-engineering quality-policy ID differs from the frozen 03B configuration.")
if quality_policy.get("status") != "frozen_feature_quality_policy" or feature_quality_config.get("status") != "frozen_feature_quality_policy":
    raise RuntimeError("The 03B feature-quality policy is not frozen.")
if feature_quality_manifest.get("status") != "frozen_feature_quality_policy":
    raise RuntimeError("The 03B feature-quality manifest is not frozen.")
expected_cutoffs = feature_quality_candidates.set_index("feature")["candidate_minimum_coverage"].astype("int16").to_dict()
if expected_cutoffs != quality_policy.get("minimum_valid_intervals") or expected_cutoffs != feature_quality_config.get("minimum_valid_intervals"):
    raise RuntimeError("Frozen feature-quality cutoffs do not match the 03B empirical evidence.")
contracts = {name: read_json(path) for name, path in CONTRACT_PATHS.items()}
expected_signatures = {
    "feature_dataset": feature_manifest["outputs"]["feature_dataset"],
    "canonical": feature_manifest["outputs"]["canonical_model_dataset"],
    "p95": feature_manifest["outputs"]["p95_model_dataset"],
    "p90": feature_manifest["outputs"]["p90_model_dataset"],
}
P95_FEATURES = feature_manifest["features"]["common"] + feature_manifest["features"]["p95_relative"] + feature_manifest["features"]["p95_event_history"]
P90_FEATURES = feature_manifest["features"]["common"] + feature_manifest["features"]["p90_relative"] + feature_manifest["features"]["p90_event_history"]
COMMON_FEATURES = feature_manifest["features"]["common"]
if not {"lag1_peak_hour_sin", "lag1_peak_hour_cos"}.issubset(COMMON_FEATURES):
    raise RuntimeError("Refreshed feature manifest is missing the policy-qualified peak-time predictors.")
if not {"target_config_id", "feature_quality_policy_id", "feature_quality_policy_status"}.issubset(contracts["feature_dataset"]["columns"]):
    raise RuntimeError("All-day feature dataset contract is missing policy provenance fields.")
input_signatures = {}
for name, path in PATHS.items():
    actual_hash = sha256(path)
    expected = expected_signatures[name]
    if actual_hash != expected["sha256"] or path.stat().st_size != expected["size_bytes"]:
        raise RuntimeError(f"{name} does not match the feature-engineering manifest")
    header = pd.read_csv(path, nrows=0).columns.tolist()
    if header != list(contracts[name]["columns"]): raise RuntimeError(f"{name} header differs from its dtype contract")
    input_signatures[name] = {"path": str(path), "sha256": actual_hash, "bytes": path.stat().st_size,
                              "rows_manifest": expected["rows"], "columns_manifest": expected["columns"]}
print("All four M0 CSVs match producer signatures and dtype contracts.")
print(f"Frozen target policy: {target_config['configuration_id']}")
print(f"Feature-quality policy: {quality_policy['policy_id']} [{quality_policy['status']}]")
print(f"Candidate cutoffs: {quality_policy['minimum_valid_intervals']}")


## Stage 1 — input, target, and population checks

The large all-calendar feature file is read in chunks. Model-ready P95, P90, and canonical tables are then loaded for the descriptive analysis. No split fields should exist yet. P95 and P90 must retain the feature-quality policy ID, status, and cutoffs recorded by feature engineering.


In [ ]:
feature_contract = contracts["feature_dataset"]
feature_rows = 0
feature_row_ids = set()
feature_duplicates = 0
feature_target_config_ids = set()
feature_quality_policy_ids = set()
feature_quality_policy_statuses = set()
for chunk in read_contract_csv(FEATURE_PATH, feature_contract, usecols=["row_id", "target_config_id", "feature_quality_policy_id", "feature_quality_policy_status"], chunksize=CSV_CHUNK_SIZE):
    values = chunk["row_id"].astype(str)
    feature_duplicates += int(values.duplicated().sum())
    feature_duplicates += sum(value in feature_row_ids for value in values)
    feature_row_ids.update(values)
    feature_target_config_ids.update(chunk["target_config_id"].astype(str).dropna().unique().tolist())
    feature_quality_policy_ids.update(chunk["feature_quality_policy_id"].astype(str).dropna().unique().tolist())
    feature_quality_policy_statuses.update(chunk["feature_quality_policy_status"].astype(str).dropna().unique().tolist())
    feature_rows += len(chunk)

canonical = read_contract_csv(CANONICAL_PATH, contracts["canonical"])
p95 = read_contract_csv(P95_PATH, contracts["p95"])
p90 = read_contract_csv(P90_PATH, contracts["p90"])
for frame in (canonical, p95, p90):
    frame["household_id"] = frame["household_id"].astype("string")
    frame["measurement_date"] = pd.to_datetime(frame["measurement_date"], errors="raise").dt.normalize()

split_master = pd.read_csv(SPLIT_MASTER_PATH, dtype={"row_id":"string","household_id":"string"})
split_p95_sig = split_manifest["input_signatures"]["p95_model_dataset"]
split_p90_sig = split_manifest["input_signatures"]["p90_model_dataset"]
p95_bytes_match_frozen_split = P95_PATH.stat().st_size == split_p95_sig["size_bytes"] and sha256(P95_PATH) == split_p95_sig["sha256"]
p90_bytes_match_frozen_split = P90_PATH.stat().st_size == split_p90_sig["size_bytes"] and sha256(P90_PATH) == split_p90_sig["sha256"]
master_row_ids_match = set(split_master.row_id.astype(str)) == set(p95.row_id.astype(str)) and split_master.row_id.is_unique and p95.row_id.is_unique
p95_rows_match_master = master_row_ids_match
split_cutoff = split_manifest["selected_cutoff_inclusive_for_development"]

input_checks = pd.DataFrame([
    ("all_day_feature_rows", feature_rows == feature_manifest["outputs"]["feature_dataset"]["rows"], f"{feature_rows:,}"),
    ("all_day_row_ids_unique", feature_duplicates == 0 and len(feature_row_ids) == feature_rows, f"duplicates={feature_duplicates:,}"),
    ("all_day_rows_use_frozen_target_config", feature_target_config_ids == {target_config["configuration_id"]}, str(sorted(feature_target_config_ids))),
    ("all_day_rows_use_frozen_quality_policy", feature_quality_policy_ids == {quality_policy["configuration_id"]} and feature_quality_policy_statuses == {quality_policy["status"]}, f"{sorted(feature_quality_policy_ids)}; {sorted(feature_quality_policy_statuses)}"),
    ("03b_frozen_policy_is_reconciled", feature_quality_manifest.get("validation") is not None and all(feature_quality_manifest["validation"].values()) and expected_cutoffs == quality_policy["minimum_valid_intervals"], str(expected_cutoffs)),
    ("frozen_p95_p90_model_dataset_hashes_unchanged", p95_bytes_match_frozen_split and p90_bytes_match_frozen_split, "byte-identical model-ready inputs to frozen split/training"),
    ("frozen_split_row_ids_and_p95_labels_unchanged", master_row_ids_match and p95_rows_match_master, f"{len(split_master):,} frozen rows; cutoff={split_cutoff}"),
    ("canonical_population_matches_manifest", len(canonical) == feature_manifest["population"]["model_rows"], f"{len(canonical):,}"),
    ("p95_p90_population_matches", len(p95) == len(p90) == len(canonical), f"{len(p95):,}/{len(p90):,}/{len(canonical):,}"),
    ("p95_p90_ordered_row_ids_match", p95["row_id"].equals(p90["row_id"]) and p95["row_id"].equals(canonical["row_id"]), "identical ordered row_id"),
    ("p95_p90_household_date_keys_match", p95["household_id"].equals(p90["household_id"]) and p95["measurement_date"].equals(p90["measurement_date"]), "same household-date observations"),
    ("all_model_ready_keys_unique", all(frame["row_id"].is_unique for frame in [canonical, p95, p90]), "unique row_id per dataset"),
    ("no_split_assigned_yet", all("split_name" not in frame.columns and "outer_split" not in frame.columns for frame in [canonical, p95, p90]), "split design is downstream"),
    ("P95_event_count_matches", int(p95["target"].sum()) == stage3_manifest["target_results"]["p95_positive_events"], str(int(p95["target"].sum()))),
    ("P90_event_count_matches", int(p90["target"].sum()) == stage3_manifest["target_results"]["p90_positive_events"], str(int(p90["target"].sum()))),
    ("no_weather_predictors", not any(any(token in name.lower() for token in ["weather", "temperature", "humidity", "precip"]) for name in P95_FEATURES + P90_FEATURES), "M0 without weather"),
], columns=["check", "passed", "evidence"])
save_table(input_checks, "03_01_input_validation.csv", "Model-ready input checks")
if not input_checks["passed"].all(): raise RuntimeError(f"EDA input validation failed: {input_checks.loc[~input_checks.passed, 'check'].tolist()}")



## Stage 2 — target prevalence and P95/P90 relationship

P95 remains the primary target. P90 is reported as a separate sensitivity target, not merged into a three-class task.


In [ ]:
def target_distribution(frame, label):
    counts = frame["target"].value_counts().reindex([0, 1], fill_value=0)
    return pd.DataFrame({"target": label, "class": ["normal", "event"], "rows": counts.to_numpy(), "rate": counts.to_numpy() / len(frame)})

class_distribution = pd.concat([target_distribution(p95, "P95 primary"), target_distribution(p90, "P90 sensitivity")], ignore_index=True)
save_table(class_distribution, "03_02_target_class_distribution.csv", "P95/P90 class distribution")

overlap = canonical[["target_high_demand_p95", "target_high_demand_p90"]].copy()
overlap["category"] = np.select(
    [overlap.target_high_demand_p95.eq(0) & overlap.target_high_demand_p90.eq(0),
     overlap.target_high_demand_p95.eq(0) & overlap.target_high_demand_p90.eq(1),
     overlap.target_high_demand_p95.eq(1) & overlap.target_high_demand_p90.eq(1)],
    ["normal_under_both", "P90_only_event", "event_under_both"], default="invalid_P95_without_P90")
overlap_summary = overlap["category"].value_counts().rename_axis("category").reset_index(name="rows")
overlap_summary["rate"] = overlap_summary["rows"] / len(overlap)
save_table(overlap_summary, "03_03_target_overlap.csv", "P95/P90 event overlap")

fig, ax = plt.subplots(figsize=(7, 4))
rates = class_distribution.loc[class_distribution["class"].eq("event")]
bars = ax.bar(rates["target"], rates["rate"] * 100, color=["#2563eb", "#f59e0b"])
ax.set_ylabel("Positive rows (%)")
ax.set_title("Household-day high-demand target prevalence")
ax.bar_label(bars, fmt="%.2f%%")
save_display_figure(fig, "03_01_target_prevalence.png")


## Stage 3 — calendar coverage and household participation

These are distributions of target-eligible household-days, not a random or row-wise train/test partition. Missing calendar months are left absent rather than filled with fabricated model rows.


In [ ]:
monthly = canonical.assign(month=canonical["measurement_date"].dt.to_period("M").astype(str)).groupby("month", as_index=False).agg(
    rows=("row_id", "size"), households=("household_id", "nunique"),
    p95_events=("target_high_demand_p95", "sum"), p95_rate=("target_high_demand_p95", "mean"),
    p90_events=("target_high_demand_p90", "sum"), p90_rate=("target_high_demand_p90", "mean"),
)
save_table(monthly, "03_04_monthly_target_prevalence.csv", "Monthly eligible population and prevalence")

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].bar(monthly["month"], monthly["rows"], color="#2563eb")
axes[0].set_ylabel("Eligible household-days")
axes[0].set_title("Eligible sample volume by month")
axes[1].plot(monthly["month"], monthly["p95_rate"] * 100, marker="o", label="P95 primary", color="#2563eb")
axes[1].plot(monthly["month"], monthly["p90_rate"] * 100, marker="o", label="P90 sensitivity", color="#f59e0b")
axes[1].set_ylabel("Event rate (%)")
axes[1].set_title("Monthly target prevalence")
axes[1].legend()
for ax in axes: ax.tick_params(axis="x", rotation=45)
save_display_figure(fig, "03_02_monthly_sample_and_rates.png")

household = canonical.groupby("household_id", as_index=False).agg(
    first_eligible_date=("measurement_date", "min"), last_eligible_date=("measurement_date", "max"),
    eligible_days=("row_id", "size"), p95_events=("target_high_demand_p95", "sum"),
    p90_events=("target_high_demand_p90", "sum"),
)
household["calendar_span_days"] = (household["last_eligible_date"] - household["first_eligible_date"]).dt.days + 1
household["p95_event_rate"] = household["p95_events"] / household["eligible_days"]
household_summary = household[["eligible_days", "calendar_span_days", "p95_events", "p90_events", "p95_event_rate"]].describe(percentiles=[.1, .25, .5, .75, .9]).T.reset_index(names="measure")
household_summary["households"] = len(household)
save_table(household, "03_05_household_participation.csv", "Household participation table")
save_table(household_summary, "03_06_household_participation_summary.csv", "Household participation summary")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(household["eligible_days"], bins=35, color="#0f766e", edgecolor="white")
axes[0].set_xlabel("Eligible dates per household")
axes[0].set_ylabel("Households")
axes[0].set_title("Household participation volume")
axes[1].hist(household["calendar_span_days"], bins=35, color="#7c3aed", edgecolor="white")
axes[1].set_xlabel("Days from first to last eligible date")
axes[1].set_ylabel("Households")
axes[1].set_title("Household participation span")
save_display_figure(fig, "03_03_household_participation.png")


## Stage 4 — feature missingness, distributions, and redundancy

The analysis includes every registered P95/P90 predictor, including the added prior-day peak-time terms. Missing values are described against the frozen 03B coverage rules, not imputed. Positive/negative boxplots and the correlation heatmap are exploratory descriptions, not model-performance tests or causal evidence.


In [ ]:
missing_rows = []
for label, frame, columns in [("P95", p95, P95_FEATURES), ("P90", p90, P90_FEATURES)]:
    for column in columns:
        missing_rows.append({"target": label, "feature": column, "missing_rows": int(frame[column].isna().sum()),
                             "missing_rate": float(frame[column].isna().mean())})
feature_missingness = pd.DataFrame(missing_rows).sort_values(["target", "missing_rate"], ascending=[True, False])
save_table(feature_missingness, "03_07_feature_missingness.csv", "Feature missingness for every predictor")

policy_to_predictor = {
    "daily_peak_kw": "lag1_peak_kw",
    "daily_mean_kw": "lag1_mean_kw",
    "daily_std_kw": None,
    "import_active_ratio": "lag1_import_active_ratio",
    "load_factor": "lag1_load_factor",
    "interval_p95_kw": "lag1_interval_p95_kw",
    "peak_time_minutes": "lag1_peak_hour_sin",
}
quality_audit_rows = []
for candidate in feature_quality_candidates.itertuples(index=False):
    predictor = policy_to_predictor[candidate.feature]
    missing_rate = float(p95[predictor].isna().mean()) if predictor else np.nan
    quality_audit_rows.append({
        "source_feature": candidate.feature,
        "candidate_minimum_valid_intervals": int(candidate.candidate_minimum_coverage),
        "screening_metric": candidate.screening_metric,
        "screening_tolerance": candidate.screening_tolerance,
        "criterion": candidate.criterion,
        "model_ready_predictor_proxy": predictor or "not used as a predictor",
        "p95_model_rows_missing_rate": missing_rate,
        "policy_id": quality_policy["policy_id"],
        "policy_status": quality_policy["status"],
        "candidate_decision_note": candidate.decision_note,
    })
feature_quality_audit = pd.DataFrame(quality_audit_rows)
save_table(feature_quality_audit, "03_12_feature_quality_policy_coverage.csv", "03B frozen feature-quality policy and predictor missingness")

p95_missing_plot = feature_missingness.loc[feature_missingness.target.eq("P95")].nlargest(20, "missing_rate").sort_values("missing_rate")
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(p95_missing_plot["feature"], p95_missing_plot["missing_rate"] * 100, color="#dc2626")
ax.set_xlabel("Missing eligible rows (%)")
ax.set_title("P95 predictor missingness — 20 highest")
save_display_figure(fig, "03_04_p95_feature_missingness.png")

distribution_features = ["lag1_peak_kw", "lag1_mean_kw", "lag1_interval_p95_kw", "lag1_peak_over_p95", "peak_mean_7d_over_30d", "p95_high_rate_30d", "lag1_peak_hour_sin", "lag1_peak_hour_cos"]
feature_distribution_rows = []
for column in distribution_features:
    if column not in p95.columns: continue
    for target_value, class_name in [(0, "normal"), (1, "event")]:
        values = pd.to_numeric(p95.loc[p95.target.eq(target_value), column], errors="coerce").dropna()
        feature_distribution_rows.append({"target": "P95", "feature": column, "class": class_name,
            "rows": len(values), "missing_rows": int(p95.loc[p95.target.eq(target_value), column].isna().sum()),
            "median": values.median(), "mean": values.mean(), "q25": values.quantile(.25), "q75": values.quantile(.75)})
feature_class_summary = pd.DataFrame(feature_distribution_rows)
save_table(feature_class_summary, "03_08_feature_class_summary.csv", "Selected P95 features by observed class")

fig, axes = plt.subplots(4, 2, figsize=(14, 13))
for ax, column in zip(axes.flat, distribution_features):
    if column not in p95.columns: continue
    values = [pd.to_numeric(p95.loc[p95.target.eq(label), column], errors="coerce").dropna().to_numpy() for label in [0, 1]]
    ax.boxplot(values, tick_labels=["Normal", "P95 event"], showfliers=False)
    ax.set_title(column)
    ax.set_ylabel("Value")
save_display_figure(fig, "03_05_selected_feature_distributions.png")

association_rows = []
for column in ["lag1_peak_kw", "lag1_peak_over_p95", "peak_mean_7d_over_30d", "p95_high_rate_30d"]:
    values = pd.to_numeric(p95[column], errors="coerce")
    scale = values.std()
    normal_median = values.loc[p95.target.eq(0)].median()
    event_median = values.loc[p95.target.eq(1)].median()
    association_rows.append({"feature": column, "normal_median": normal_median, "event_median": event_median,
        "median_difference_event_minus_normal": event_median - normal_median,
        "standardized_median_difference": (event_median - normal_median) / scale if pd.notna(scale) and scale > 0 else np.nan,
        "missing_rate": float(values.isna().mean()), "interpretation": "descriptive association only; not predictive validation"})
relative_association = pd.DataFrame(association_rows)
save_table(relative_association, "03_09_relative_vs_absolute_associations.csv", "Household-relative versus absolute associations")

corr_features = [name for name in ["lag1_peak_kw", "lag2_peak_kw", "peak_d7", "same_weekday_peak_mean4", "peak_mean_3d", "peak_mean_7d", "peak_mean_30d", "peak_mean_7d_over_30d", "lag1_peak_over_p95", "lag1_peak_minus_p95", "peak_mean_7d_over_p95", "p95_high_rate_30d", "lag1_load_factor", "lag1_import_active_ratio", "lag1_mean_kw", "lag1_interval_p95_kw",
                "lag1_peak_hour_sin", "lag1_peak_hour_cos", "day_of_week_sin", "month_sin"] if name in p95.columns]
corr_data = p95[corr_features].apply(lambda s: pd.to_numeric(s, errors="coerce").astype("float64"))
spearman = corr_data.corr(method="spearman")
save_table(spearman.rename_axis("feature").reset_index(), "03_10_p95_spearman_correlation.csv", "Selected P95 Spearman correlations")
fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(spearman.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
ax.set_xticks(range(len(corr_features)), corr_features, rotation=90)
ax.set_yticks(range(len(corr_features)), corr_features)
ax.set_title("Spearman correlation among selected P95 predictors")
fig.colorbar(image, ax=ax, label="Spearman correlation")
save_display_figure(fig, "03_06_p95_spearman_correlation.png")


## Stage 5 — final EDA validation and handoff

This stage makes no split and does not transform the saved M0 predictors. Its summaries and figures are reports only. `04_split_design.ipynb` follows this EDA and is the only notebook that will assign `outer_split`.


In [ ]:
policy_provenance = pd.DataFrame([
    {"policy": "target", "configuration_id": target_config["configuration_id"], "status": target_config["status"]},
    {"policy": "historical_feature_quality", "configuration_id": feature_quality_config["configuration_id"], "status": feature_quality_config["status"]},
])
save_table(policy_provenance, "03_00_research_policy_provenance.csv", "Frozen research-policy provenance")

required_tables = [
    "03_00_research_policy_provenance.csv",
    "03_01_input_validation.csv", "03_02_target_class_distribution.csv", "03_03_target_overlap.csv",
    "03_04_monthly_target_prevalence.csv", "03_05_household_participation.csv", "03_06_household_participation_summary.csv",
    "03_07_feature_missingness.csv", "03_08_feature_class_summary.csv", "03_09_relative_vs_absolute_associations.csv",
    "03_10_p95_spearman_correlation.csv", "03_12_feature_quality_policy_coverage.csv",
]
required_figures = [
    "03_01_target_prevalence.png", "03_02_monthly_sample_and_rates.png", "03_03_household_participation.png",
    "03_04_p95_feature_missingness.png", "03_05_selected_feature_distributions.png", "03_06_p95_spearman_correlation.png",
]
final_checks = pd.DataFrame([
    ("all_input_checks_pass", bool(input_checks.passed.all()), str(len(input_checks))),
    ("P95_is_primary_and_P90_is_sensitivity", feature_manifest["features"] is not None and len(P95_FEATURES) > 0 and len(P90_FEATURES) > 0, "separate binary tasks"),
    ("identical_P95_P90_observations", p95.row_id.equals(p90.row_id), f"{len(p95):,} rows"),
    ("no_split_or_model_training", all("split_name" not in frame.columns and "outer_split" not in frame.columns for frame in [canonical, p95, p90]), "EDA-only stage"),
    ("all_feature_missingness_reported", set(feature_missingness.feature) == set(P95_FEATURES) | set(P90_FEATURES), f"{feature_missingness.feature.nunique()} distinct features"),
    ("all_required_tables_exist", all((TABLE_DIR / name).exists() and (TABLE_DIR / name).stat().st_size > 0 for name in required_tables), str([name for name in required_tables if not (TABLE_DIR / name).exists()])),
    ("all_required_figures_exist", all((FIGURE_DIR / name).exists() and (FIGURE_DIR / name).stat().st_size > 0 for name in required_figures), str([name for name in required_figures if not (FIGURE_DIR / name).exists()])),
    ("all_report_tables_are_csv", all(path.suffix.lower() == ".csv" for path in TABLE_DIR.glob("*.csv")), str(sorted(path.suffix for path in TABLE_DIR.glob("*.csv")))),
    ("target_policy_status_propagated", feature_manifest["target_policy_status"] == stage3_manifest["target_policy_status"] == target_config["status"], feature_manifest["target_policy_status"]),
    ("feature_quality_policy_provenance_reconciled", quality_policy["configuration_id"] in feature_quality_policy_ids and quality_policy["status"] in feature_quality_policy_statuses and expected_cutoffs == quality_policy["minimum_valid_intervals"], quality_policy["configuration_id"]),
    ("frozen_model_dataset_hashes_and_split_row_ids_reused", p95_bytes_match_frozen_split and p90_bytes_match_frozen_split and master_row_ids_match and p95_rows_match_master, f"cutoff={split_cutoff}; model-ready bytes unchanged"),
    ("new_peak_time_predictors_reported", {"lag1_peak_hour_sin", "lag1_peak_hour_cos"}.issubset(set(P95_FEATURES) & set(P90_FEATURES)), str([x for x in COMMON_FEATURES if "peak_hour" in x])),
    ("feature_quality_audit_rows_complete", set(feature_quality_audit["source_feature"]) == set(feature_quality_candidates["feature"]), f"{len(feature_quality_audit)} candidate rules"),
], columns=["check", "passed", "evidence"])
save_table(final_checks, "03_11_final_validation.csv", "Model-ready EDA final checks")
if not final_checks.passed.all(): raise RuntimeError(f"Final EDA checks failed: {final_checks.loc[~final_checks.passed, 'check'].tolist()}")

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "stage": "03_model_ready_eda",
    "input_signatures": input_signatures, "target_policy_status": feature_manifest["target_policy_status"],
    "target_configuration_id": target_config["configuration_id"],
    "feature_quality_policy": quality_policy,
    "feature_quality_configuration_id": feature_quality_config["configuration_id"],
    "frozen_split": {"manifest_path": str(SPLIT_MANIFEST_PATH), "cutoff_inclusive_for_development": str(split_cutoff), "master_rows": int(len(split_master)), "model_dataset_hashes_unchanged": bool(p95_bytes_match_frozen_split and p90_bytes_match_frozen_split)},
    "feature_quality_policy_source_manifest": str(FEATURE_QUALITY_MANIFEST_PATH),
    "population": {"model_rows": len(canonical), "households": canonical.household_id.nunique(),
                   "first_target_date": str(canonical.measurement_date.min().date()), "last_target_date": str(canonical.measurement_date.max().date()),
                   "p95_positive_events": int(p95.target.sum()), "p90_positive_events": int(p90.target.sum())},
    "split_created": False, "model_fitted": False,
    "reports": {"tables": required_tables + ["03_11_final_validation.csv"], "figures": required_figures},
    "validation": {row.check: bool(row.passed) for row in final_checks.itertuples(index=False)},
    "software": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
}
(TABLE_DIR / "03_model_ready_eda_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
print(f"Target config: {target_config['configuration_id']}")
print(f"Feature-quality config: {feature_quality_config['configuration_id']}")
print("Existing split and model-ready feature hashes verified unchanged; no new split was created and no model was fitted.")
print(f"Tables: {TABLE_DIR}")
print(f"Figures: {FIGURE_DIR}")


### Handoff

The frozen target and feature-quality policy IDs are recorded in the provenance table and manifest. This notebook verified that both model-ready datasets remain byte-identical to the inputs recorded by the existing frozen split manifest; keep that split and assignment unchanged.
